# 03 - Residual Stream Interventions

Runs the core experimental conditions:
- **self_prefill**: Re-tokenize COT, fresh forward pass, sample answer (sanity check)
- **zeroed_layer_k**: Replace residual at last position with raw embedding at layer k

**IMPORTANT**: Restart the kernel before running this notebook to free GPU memory from vLLM (notebook 02).

**Set `CONDITION` in the cell below** to control which condition runs.

**Fully resumable** - caches one JSON file per (condition, problem) in `cache/interventions/`.

In [1]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [ ]:
# Verify GPU is free and purge errored cache files
import torch, json

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    if free < 50e9:
        print("WARNING: Less than 50GB free. Restart the kernel to free vLLM/other memory!")
        print("This notebook WILL fail with OOM errors if you proceed.")

# Purge only errored cache files (so they get retried)
purged = 0
for p in INTERVENTION_CACHE.glob("*.json"):
    try:
        entry = json.loads(p.read_text())
        if entry.get("error") is not None:
            p.unlink()
            purged += 1
    except Exception:
        p.unlink()
        purged += 1

print(f"Purged {purged} errored cache files (keeping {len(list(INTERVENTION_CACHE.glob('*.json')))} valid)")

In [3]:
# =============================================
# SET THE CONDITION TO RUN HERE
# =============================================
# Options:
#   "self_prefill"
#   "zeroed_layer_0"
#   "zeroed_layer_9"
#   "zeroed_layer_18"
#   "zeroed_layer_27"
#   "zeroed_layer_35"
#
# Or set to "all" to run all conditions sequentially.

CONDITION = "all"

In [ ]:
import json
import traceback
import torch
from tqdm.auto import tqdm
from lib.data import build_prefill_string
from lib.intervention import load_model, generate_answer

In [ ]:
# Load model (uses transformers + hooks for generate() support)
model, tokenizer = load_model(MODEL_NAME)

In [6]:
# Load COT results (from notebook 02)
cots_path = RESULTS_DIR / "cots.jsonl"
assert cots_path.exists(), f"Run 02_generate_cots.ipynb first. Missing: {cots_path}"

cots = []
with open(cots_path) as f:
    for line in f:
        entry = json.loads(line)
        if entry["error"] is None and entry["cot_text"] is not None:
            cots.append(entry)

print(f"Loaded {len(cots)} valid COT results")

# Check that full_response is available
has_full = sum(1 for c in cots if c.get("full_response"))
print(f"  with full_response: {has_full}/{len(cots)}")

Loaded 256 valid COT results
  with full_response: 256/256


In [ ]:
def parse_condition(condition_name: str) -> int | None:
    """Parse condition name to get the layer index for zeroing.
    Returns None for self_prefill (no intervention).
    """
    if condition_name == "self_prefill":
        return None
    elif condition_name.startswith("zeroed_layer_"):
        return int(condition_name.split("_")[-1])
    else:
        raise ValueError(f"Unknown condition: {condition_name}")


def run_condition(condition_name: str, cots: list[dict]):
    """Run a single experimental condition across all COT results."""
    zero_at_layer = parse_condition(condition_name)
    
    # Resume logic: strip "{condition_name}_" prefix from filename to get problem_id
    prefix = f"{condition_name}_"
    done_ids = set()
    for p in INTERVENTION_CACHE.glob(f"{prefix}*.json"):
        pid_str = p.stem[len(prefix):]
        done_ids.add(int(pid_str))

    todo = [c for c in cots if c["problem_id"] not in done_ids]
    print(f"\n[{condition_name}] Resuming: {len(done_ids)} done, {len(todo)} remaining")
    
    if not todo:
        return
    
    correct = 0
    total = 0
    
    for entry in tqdm(todo, desc=condition_name):
        try:
            prefill = build_prefill_string(entry["question"], entry["full_response"])
            result = generate_answer(prefill, zero_at_layer=zero_at_layer)
            
            cache_entry = {
                "problem_id": entry["problem_id"],
                "question": entry["question"],
                "gold_answer": entry["gold_answer"],
                "condition": condition_name,
                "cot_text": entry["cot_text"],
                "generated_text": result["generated_text"],
                "predicted_answer": result["predicted_answer"],
                "error": None,
            }
            
            if result["predicted_answer"] == entry["gold_answer"]:
                correct += 1
            total += 1
            
        except Exception:
            cache_entry = {
                "problem_id": entry["problem_id"],
                "question": entry["question"],
                "gold_answer": entry["gold_answer"],
                "condition": condition_name,
                "cot_text": entry["cot_text"],
                "generated_text": None,
                "predicted_answer": None,
                "error": traceback.format_exc(),
            }
        
        fname = f"{prefix}{entry['problem_id']}.json"
        (INTERVENTION_CACHE / fname).write_text(json.dumps(cache_entry))
    
    if total > 0:
        print(f"[{condition_name}] Batch accuracy: {correct}/{total} = {correct/total:.1%}")

In [8]:
# Run the selected condition(s)
if CONDITION == "all":
    conditions_to_run = CONDITIONS
else:
    conditions_to_run = [CONDITION]

print(f"Conditions to run: {conditions_to_run}")

for cond in conditions_to_run:
    run_condition(cond, cots)

Conditions to run: ['self_prefill', 'zeroed_layer_0', 'zeroed_layer_1', 'zeroed_layer_2', 'zeroed_layer_3', 'zeroed_layer_4', 'zeroed_layer_5', 'zeroed_layer_6', 'zeroed_layer_7', 'zeroed_layer_8', 'zeroed_layer_9', 'zeroed_layer_10', 'zeroed_layer_11', 'zeroed_layer_12', 'zeroed_layer_13', 'zeroed_layer_14', 'zeroed_layer_15', 'zeroed_layer_16', 'zeroed_layer_17', 'zeroed_layer_18', 'zeroed_layer_19', 'zeroed_layer_20', 'zeroed_layer_21', 'zeroed_layer_22', 'zeroed_layer_23', 'zeroed_layer_24', 'zeroed_layer_25', 'zeroed_layer_26', 'zeroed_layer_27', 'zeroed_layer_28', 'zeroed_layer_29', 'zeroed_layer_30', 'zeroed_layer_31', 'zeroed_layer_32', 'zeroed_layer_33', 'zeroed_layer_34', 'zeroed_layer_35']

[self_prefill] Resuming: 0 done, 256 remaining


self_prefill:   0%|          | 0/256 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[self_prefill] Batch accuracy: 1/1 = 100.0%

[zeroed_layer_0] Resuming: 0 done, 256 remaining


zeroed_layer_0:   0%|          | 0/256 [00:00<?, ?it/s]

...

[zeroed_layer_33] Batch accuracy: 1/1 = 100.0%

[zeroed_layer_34] Resuming: 0 done, 256 remaining


zeroed_layer_34:   0%|          | 0/256 [00:00<?, ?it/s]

[zeroed_layer_34] Batch accuracy: 1/1 = 100.0%

[zeroed_layer_35] Resuming: 0 done, 256 remaining


zeroed_layer_35:   0%|          | 0/256 [00:00<?, ?it/s]

[zeroed_layer_35] Batch accuracy: 0/1 = 0.0%


In [9]:
# Aggregate results per condition into results/*.jsonl
for cond in CONDITIONS:
    prefix = f"{cond}_"
    entries = []
    for p in sorted(INTERVENTION_CACHE.glob(f"{prefix}*.json"), key=lambda x: int(x.stem[len(prefix):])):
        entries.append(json.loads(p.read_text()))
    
    if not entries:
        continue
    
    output_path = RESULTS_DIR / f"{cond}.jsonl"
    with open(output_path, "w") as f:
        for entry in entries:
            f.write(json.dumps(entry) + "\n")
    
    valid = [e for e in entries if e["error"] is None]
    correct = sum(1 for e in valid if e["predicted_answer"] == e["gold_answer"])
    errors = len(entries) - len(valid)
    
    if valid:
        print(f"{cond}: {correct}/{len(valid)} correct ({correct/len(valid):.1%}), {errors} errors -> {output_path}")
    else:
        print(f"{cond}: 0 valid, {errors} errors -> {output_path}")

self_prefill: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/self_prefill.jsonl
zeroed_layer_0: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_0.jsonl
zeroed_layer_1: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_1.jsonl
zeroed_layer_2: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_2.jsonl
zeroed_layer_3: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_3.jsonl
zeroed_layer_4: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_4.jsonl
zeroed_layer_5: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_5.jsonl
zeroed_layer_6: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_6.jsonl
zeroed_layer_7: 1/1 correct (100.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_layer_7.jsonl
zeroed_layer_8: 0/1 correct (0.0%), 255 errors -> /workspace/10-4-2026/results/zeroed_

In [10]:
# Quick summary of all conditions
import re

print("\n" + "="*60)
print("SUMMARY")
print("="*60)

# Recompute normal accuracy from full_response using the same extraction method
# as the intervention conditions (find "The answer is: X" in full_response)
normal_correct = 0
normal_total = 0
for c in cots:
    if c.get("error"):
        continue
    fr = c.get("full_response", "")
    matches = list(re.finditer(r"[Tt]he answer is:?\s*(-?\d[\d,]*)", fr))
    if matches:
        predicted = int(matches[-1].group(1).replace(",", ""))
    else:
        nums = re.findall(r"-?\d[\d,]*", fr)
        predicted = int(nums[-1].replace(",", "")) if nums else None
    if predicted == c["gold_answer"]:
        normal_correct += 1
    normal_total += 1

print(f"{'normal':<25} {normal_correct}/{normal_total} = {normal_correct/normal_total:.1%}")

for cond in CONDITIONS:
    result_path = RESULTS_DIR / f"{cond}.jsonl"
    if not result_path.exists():
        print(f"{cond:<25} not yet run")
        continue
    
    entries = [json.loads(l) for l in result_path.read_text().splitlines()]
    valid = [e for e in entries if e["error"] is None]
    correct = sum(1 for e in valid if e["predicted_answer"] == e["gold_answer"])
    print(f"{cond:<25} {correct}/{len(valid)} = {correct/len(valid):.1%}")


SUMMARY
normal                    208/256 = 81.2%
self_prefill              1/1 = 100.0%
zeroed_layer_0            1/1 = 100.0%
zeroed_layer_1            1/1 = 100.0%
zeroed_layer_2            1/1 = 100.0%
zeroed_layer_3            1/1 = 100.0%
zeroed_layer_4            1/1 = 100.0%
zeroed_layer_5            1/1 = 100.0%
zeroed_layer_6            1/1 = 100.0%
zeroed_layer_7            1/1 = 100.0%
zeroed_layer_8            0/1 = 0.0%
zeroed_layer_9            1/1 = 100.0%
zeroed_layer_10           0/1 = 0.0%
zeroed_layer_11           1/1 = 100.0%
zeroed_layer_12           1/1 = 100.0%
zeroed_layer_13           1/1 = 100.0%
zeroed_layer_14           0/1 = 0.0%
zeroed_layer_15           1/1 = 100.0%
zeroed_layer_16           1/1 = 100.0%
zeroed_layer_17           1/1 = 100.0%
zeroed_layer_18           1/1 = 100.0%
zeroed_layer_19           0/1 = 0.0%
zeroed_layer_20           1/1 = 100.0%
zeroed_layer_21           0/1 = 0.0%
zeroed_layer_22           1/1 = 100.0%
zeroed_layer_23        

In [11]:
# Diagnostic: inspect self_prefill predictions vs gold
import json

sp_path = RESULTS_DIR / "self_prefill.jsonl"
cots_path = RESULTS_DIR / "cots.jsonl"

sp_entries = [json.loads(l) for l in sp_path.read_text().splitlines()]
cot_entries = {e["problem_id"]: e for e in [json.loads(l) for l in cots_path.read_text().splitlines()]}

print("Self-prefill: first 10 predictions vs gold")
print("-" * 70)
for e in sp_entries[:10]:
    cot = cot_entries.get(e["problem_id"], {})
    print(f"PID {e['problem_id']:>3} | gold={e['gold_answer']:<8} | predicted={str(e['predicted_answer']):<8} | top1='{e['top1_token']}' (p={e['top1_prob']})")
    # Show the end of the COT text
    if cot.get("cot_text"):
        print(f"         COT ends with: ...{cot['cot_text'][-80:]}")
    if cot.get("answer_portion"):
        print(f"         Answer portion: {cot['answer_portion'][:100]}")
    print()

Self-prefill: first 10 predictions vs gold
----------------------------------------------------------------------
PID   0 | gold=72       | predicted=72       | top1='7' (p=0.82967)
         COT ends with: ...makes 72. Yeah, that seems right. So the total is 72. I think that's the answer.
         Answer portion: Natalia sold 48 clips in April. In May, she sold half as many, which is $ \frac{48}{2} = 24 $ clips.

PID   1 | gold=10       | predicted=None     | top1='None' (p=None)
         COT ends with: ...0 = $10. Yep, same result. So I think that's right. So the answer should be $10.
         Answer portion: Weng earns $12 per hour. To find her earnings for 50 minutes, convert minutes to hours:  
50 minutes

PID   2 | gold=5        | predicted=None     | top1='None' (p=None)
         COT ends with: ...ives 95. Then 100 - 95 is 5. Yeah, that seems right. So the answer should be $5.
         Answer portion: Betty needs $100 for the wallet. She already has half of that, which is $50. He